# FER2013 Facial-Expression Classification

**Assignment 1 | Healthcare diagnostics or education research**

This notebook compares an MLP and a CNN on FER2013. The supplied file is the original FER2013 data, not the FER+ relabelled version named in the brief. Descriptive EDA, including the class counts, sample grid, and class-wise mean images, uses the full CSV, including PublicTest and PrivateTest. Those partitions are excluded from training and checkpoint selection; PublicTest is used for the final comparison only after model selection, and PrivateTest is not used for performance evaluation.

The local archive was downloaded from the Kaggle FER2013 repository (https://www.kaggle.com/datasets/deadskull7/fer2013) on 25 August 2026. The executed notebook records Python 3.10.16; the course specification lists Python 3.11.8.


In [1]:
# Core imports and reproducibility settings.
from pathlib import Path
import json
import random
import sys
import platform
import time
import numpy as np
sys.modules["numexpr"] = None  # Skip an incompatible optional accelerator.
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
# Load transforms; define the missing optional NMS schema in this local build.
try:
    from torchvision import transforms
except RuntimeError as exc:
    if "torchvision::nms" not in str(exc):
        raise
    _torchvision_lib = torch.library.Library("torchvision", "DEF")
    _torchvision_lib.define("nms(Tensor dets, Tensor scores, float iou_threshold) -> Tensor")
    from torchvision import transforms

ROOT = Path.cwd()
DATA_PATH = ROOT / "fer2013.csv"
DATA_SOURCE_URL = "https://www.kaggle.com/datasets/deadskull7/fer2013"
DATA_DOWNLOAD_DATE = "2026-08-25"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
LABEL_NAMES = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
print({"device": str(DEVICE), "python": platform.python_version(), "data_file": str(DATA_PATH), "data_source": DATA_SOURCE_URL, "download_date": DATA_DOWNLOAD_DATE, "seed": SEED})


{'device': 'mps', 'python': '3.10.16', 'data_file': '/Users/sidslaptop/Documents/my docs/source/multi-source/ass1/fer2013.csv', 'data_source': 'https://www.kaggle.com/datasets/deadskull7/fer2013', 'download_date': '2026-08-25', 'seed': 42}


In [2]:
# Load the CSV and validate its schema.
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Extract fer2013.csv.zip first.")
df = pd.read_csv(DATA_PATH)
required = {"emotion", "pixels", "Usage"}
if set(df.columns) != required:
    raise ValueError(f"Unexpected columns: {df.columns.tolist()}")

pixel_lengths = df["pixels"].str.split().str.len()
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().to_dict())
print("Pixels per row:", pixel_lengths.unique().tolist())
print("Usage counts:", df["Usage"].value_counts().to_dict())
print("Emotion counts:", df["emotion"].value_counts().sort_index().to_dict())


Shape: (35887, 3)
Missing values: {'emotion': 0, 'pixels': 0, 'Usage': 0}
Pixels per row: [2304]


Usage counts: {'Training': 28709, 'PublicTest': 3589, 'PrivateTest': 3589}
Emotion counts: {0: 4953, 1: 547, 2: 5121, 3: 8989, 4: 6077, 5: 4002, 6: 6198}


In [3]:
# Plot class balance and representative images.
counts = df["emotion"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(LABEL_NAMES, counts.values, color="#315f8a")
ax.set_title("FER2013 class distribution")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=25)
ax.bar_label(bars, fmt="%d", padding=3, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_distribution.png", dpi=180)
plt.close(fig)

sample_rows = []
for label in range(7):
    sample_rows.append(df[df["emotion"] == label].sample(2, random_state=SEED + label))
sample_df = pd.concat(sample_rows).reset_index(drop=True)
fig, axes = plt.subplots(2, 7, figsize=(12, 4.2))
for i, (_, row) in enumerate(sample_df.iterrows()):
    axes.flat[i].imshow(np.asarray(row["pixels"].split(), dtype=np.uint8).reshape(48, 48), cmap="gray")
    axes.flat[i].set_title(LABEL_NAMES[int(row["emotion"])], fontsize=9)
    axes.flat[i].axis("off")
fig.suptitle("Representative FER2013 images", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "sample_grid.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Show class-wise mean images as a second EDA view.
images_for_eda = np.asarray([np.fromstring(s, dtype=np.uint8, sep=" ") for s in df["pixels"]], dtype=np.uint8).reshape(-1, 48, 48)
mean_images = np.stack([images_for_eda[df["emotion"].to_numpy() == label].mean(axis=0) / 255.0 for label in range(7)])
fig, axes = plt.subplots(1, 7, figsize=(12, 2.2))
for label, ax in enumerate(axes):
    ax.imshow(mean_images[label], cmap="gray", vmin=0, vmax=1)
    ax.set_title(LABEL_NAMES[label], fontsize=8)
    ax.axis("off")
fig.suptitle("Class-wise mean images", y=1.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_mean_images.png", dpi=180, bbox_inches="tight")
plt.close(fig)


In [4]:
# Convert pixels to 48x48 grayscale arrays and create stratified splits.
images = np.asarray([np.fromstring(s, dtype=np.uint8, sep=" ") for s in df["pixels"]], dtype=np.uint8)
images = images.reshape(-1, 1, 48, 48)
labels = df["emotion"].to_numpy(dtype=np.int64)
usages = df["Usage"].to_numpy()
train_pool = np.flatnonzero(usages == "Training")
test_idx = np.flatnonzero(usages == "PublicTest")
train_idx, val_idx = [], []
rng = np.random.default_rng(SEED)
for label in range(7):
    label_idx = train_pool[labels[train_pool] == label]
    rng.shuffle(label_idx)
    cut = int(len(label_idx) * 0.9)
    train_idx.extend(label_idx[:cut])
    val_idx.extend(label_idx[cut:])
train_idx, val_idx = np.asarray(train_idx), np.asarray(val_idx)
rng.shuffle(train_idx)
rng.shuffle(val_idx)
print({"train": len(train_idx), "validation": len(val_idx), "public_test": len(test_idx)})


{'train': 25835, 'validation': 2874, 'public_test': 3589}


In [5]:
# Use tensor transforms for training-time variation and stable scaling.
class FERDataset(Dataset):
    def __init__(self, indices, transform=None):
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        idx = int(self.indices[position])
        x = torch.from_numpy(images[idx].copy()).float().div_(255.0)
        if self.transform is not None:
            x = self.transform(x)
        y = torch.tensor(labels[idx], dtype=torch.long)
        return x, y

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10, fill=0),
])
train_loader = DataLoader(FERDataset(train_idx, train_transform), batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(FERDataset(val_idx), batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(FERDataset(test_idx), batch_size=256, shuffle=False, num_workers=0)
class_counts = np.bincount(labels[train_idx], minlength=7)
print("Training class counts:", class_counts.astype(int).tolist())
print("Training augmentation: horizontal flip p=0.5; rotation +/-10 degrees")


Training class counts: [3595, 392, 3687, 6493, 4347, 2853, 4468]
Training augmentation: horizontal flip p=0.5; rotation +/-10 degrees


In [6]:
# Define a flat-pixel baseline and a small convolutional model.
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(48 * 48, 128), nn.ReLU(), nn.Dropout(0.25), nn.Linear(128, 7)
        )

    def forward(self, x):
        return self.net(x)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.20),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 6 * 6, 128), nn.ReLU(), nn.Dropout(0.40), nn.Linear(128, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

PARAMETER_COUNTS = {"MLP": count_trainable_parameters(MLP()), "CNN": count_trainable_parameters(SmallCNN())}
print("Trainable parameters:", PARAMETER_COUNTS)

def make_model(name):
    return (MLP() if name == "MLP" else SmallCNN()).to(DEVICE)

def metric_dict(y_true, y_pred):
    cm = np.zeros((7, 7), dtype=np.int64)
    for actual, predicted in zip(y_true, y_pred):
        cm[int(actual), int(predicted)] += 1
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, cm.sum(axis=0), out=np.zeros(7), where=cm.sum(axis=0) != 0)
    recall = np.divide(tp, cm.sum(axis=1), out=np.zeros(7), where=cm.sum(axis=1) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(7), where=(precision + recall) != 0)
    support = cm.sum(axis=1)
    accuracy = float(tp.sum() / max(cm.sum(), 1))
    return {
        "accuracy": accuracy,
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "weighted_f1": float(np.average(f1, weights=support)),
        "per_class": {LABEL_NAMES[i]: {"precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i]), "support": int(support[i])} for i in range(7)},
        "confusion_matrix": cm.tolist(),
    }

def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total = 0.0, 0
    all_true, all_pred = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * len(y)
        total += len(y)
        all_true.extend(y.detach().cpu().numpy())
        all_pred.extend(logits.argmax(1).detach().cpu().numpy())
    metrics = metric_dict(np.asarray(all_true), np.asarray(all_pred))
    metrics["loss"] = total_loss / max(total, 1)
    return metrics

def predict_dataset(model, loader):
    model.eval()
    all_images, all_true, all_pred = [], [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            all_images.append(x.cpu())
            all_true.append(y.numpy())
            all_pred.append(logits.argmax(1).cpu().numpy())
    return torch.cat(all_images).numpy(), np.concatenate(all_true), np.concatenate(all_pred)

def train_model(name, epochs):
    model = make_model(name)
    loss_fn = nn.CrossEntropyLoss()
    lr = 8e-4
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = []
    best_state, best_epoch, best_score = None, 0, -1.0
    start = time.time()
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, loss_fn, optimizer)
        val_metrics = run_epoch(model, val_loader, loss_fn)
        scheduler.step()
        history.append({"epoch": epoch, "train": train_metrics, "validation": val_metrics})
        print(f"{name} epoch {epoch}/{epochs}: train loss={train_metrics['loss']:.3f}, val loss={val_metrics['loss']:.3f}, val macro-F1={val_metrics['macro_f1']:.3f}")
        if val_metrics["macro_f1"] > best_score:
            best_score = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    test_metrics = run_epoch(model, test_loader, loss_fn)
    test_metrics["selected_epoch"] = best_epoch
    print(f"{name} best epoch={best_epoch}; test accuracy={test_metrics['accuracy']:.3f}, macro-F1={test_metrics['macro_f1']:.3f}, time={time.time()-start:.1f}s")
    return model, history, test_metrics


Trainable parameters: {'MLP': 295943, 'CNN': 730151}


In [7]:
# Train both models for an equal 15-epoch comparison with shared augmentation.
mlp_model, mlp_history, mlp_test = train_model("MLP", epochs=15)
cnn_model, cnn_history, cnn_test = train_model("CNN", epochs=15)


MLP epoch 1/15: train loss=1.806, val loss=1.757, val macro-F1=0.172


MLP epoch 2/15: train loss=1.757, val loss=1.736, val macro-F1=0.137


MLP epoch 3/15: train loss=1.733, val loss=1.701, val macro-F1=0.193


MLP epoch 4/15: train loss=1.715, val loss=1.679, val macro-F1=0.207


MLP epoch 5/15: train loss=1.709, val loss=1.696, val macro-F1=0.237


...

CNN epoch 12/15: train loss=1.319, val loss=1.170, val macro-F1=0.424


CNN epoch 13/15: train loss=1.316, val loss=1.172, val macro-F1=0.428


CNN epoch 14/15: train loss=1.305, val loss=1.169, val macro-F1=0.429


CNN epoch 15/15: train loss=1.299, val loss=1.167, val macro-F1=0.428


CNN best epoch=10; test accuracy=0.544, macro-F1=0.430, time=106.5s


In [8]:
# Plot learning curves, confusion matrices, and the model comparison.
histories = {"MLP": mlp_history, "CNN": cnn_history}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, history in histories.items():
    epochs = [h["epoch"] for h in history]
    axes[0].plot(epochs, [h["train"]["loss"] for h in history], marker="o", label=f"{name} train")
    axes[0].plot(epochs, [h["validation"]["loss"] for h in history], marker="o", linestyle="--", label=f"{name} validation")
    axes[1].plot(epochs, [h["validation"]["macro_f1"] for h in history], marker="o", label=name)
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[1].set_title("Validation macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro-F1")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "training_curves.png", dpi=180)
plt.close(fig)

def plot_cm(metrics, name):
    cm = np.asarray(metrics["confusion_matrix"])
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6.3, 5.3))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(7), LABEL_NAMES, rotation=35, ha="right")
    ax.set_yticks(range(7), LABEL_NAMES)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{name} normalised confusion matrix")
    for i in range(7):
        for j in range(7):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm_norm[i, j] > 0.55 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Row proportion")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"confusion_matrix_{name.lower()}.png", dpi=180)
    plt.close(fig)

plot_cm(mlp_test, "MLP")
plot_cm(cnn_test, "CNN")
test_images, test_true, test_pred = predict_dataset(cnn_model, test_loader)
pair_targets = [(2, 4), (4, 6), (0, 4), (2, 0), (4, 3)]
selected = []
for true_label, predicted_label in pair_targets:
    matches = np.flatnonzero((test_true == true_label) & (test_pred == predicted_label))
    selected.extend(matches[:2].tolist())
fallback = np.flatnonzero(test_true != test_pred)
for idx in fallback:
    if len(selected) >= 8:
        break
    if int(idx) not in selected:
        selected.append(int(idx))
selected = selected[:8]
fig, axes = plt.subplots(2, 4, figsize=(11, 5.7))
for ax, idx in zip(axes.flat, selected):
    ax.imshow(test_images[idx, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"True: {LABEL_NAMES[test_true[idx]]}\nPred: {LABEL_NAMES[test_pred[idx]]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Representative CNN misclassifications from PublicTest", y=1.01)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(FIG_DIR / "misclassification_grid.png", dpi=180, bbox_inches="tight")
plt.close(fig)
misclassification_examples = [{"index": int(idx), "true": LABEL_NAMES[test_true[idx]], "predicted": LABEL_NAMES[test_pred[idx]]} for idx in selected]
comparison = pd.DataFrame({
    "Model": ["MLP", "CNN"],
    "Accuracy": [mlp_test["accuracy"], cnn_test["accuracy"]],
    "Macro precision": [mlp_test["macro_precision"], cnn_test["macro_precision"]],
    "Macro recall": [mlp_test["macro_recall"], cnn_test["macro_recall"]],
    "Macro F1": [mlp_test["macro_f1"], cnn_test["macro_f1"]],
    "Weighted F1": [mlp_test["weighted_f1"], cnn_test["weighted_f1"]],
})
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
fig, ax = plt.subplots(figsize=(8, 4.5))
comparison.set_index("Model")[["Accuracy", "Macro F1", "Weighted F1"]].plot.bar(ax=ax, color=["#6baed6", "#2171b5", "#08306b"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Held-out PublicTest model comparison")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "model_comparison.png", dpi=180)
plt.close(fig)

results = {
    "device": str(DEVICE),
    "seed": SEED,
    "augmentation": {"training_only": True, "transforms": ["RandomHorizontalFlip(p=0.5)", "RandomRotation(degrees=10)"]},
    "dataset": {"name": "FER2013", "shape": list(df.shape), "image_shape": [48, 48], "labels": LABEL_NAMES, "source_file": DATA_PATH.name, "source_url": DATA_SOURCE_URL, "download_date": DATA_DOWNLOAD_DATE},
    "environment": {"python": platform.python_version(), "torch": torch.__version__},
    "parameter_counts": PARAMETER_COUNTS,
    "splits": {"train": int(len(train_idx)), "validation": int(len(val_idx)), "public_test": int(len(test_idx)), "private_test_available": int((usages == "PrivateTest").sum())},
    "class_counts": {LABEL_NAMES[i]: int(counts.iloc[i]) for i in range(7)},
    "misclassification_examples": misclassification_examples,
    "models": {"MLP": {"history": mlp_history, "test": mlp_test}, "CNN": {"history": cnn_history, "test": cnn_test}},
}
(ROOT / "results.json").write_text(json.dumps(results, indent=2))
comparison.to_csv(ROOT / "metrics.csv", index=False)
print("Saved figures, results.json, and metrics.csv")


Model  Accuracy  Macro precision  Macro recall  Macro F1  Weighted F1
  MLP     0.377            0.277         0.286     0.247        0.312
  CNN     0.544            0.434         0.447     0.430        0.519
Saved figures, results.json, and metrics.csv


## Interpretation

The CNN is stronger than the flat-pixel MLP on PublicTest, but both models perform unevenly across classes. Disgust has no correct CNN predictions. FER2013 also provides no demographic data, so subgroup performance cannot be measured. Its labels describe perceived facial appearance; they do not reveal a person's internal emotional state.
